In [8]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
print(jax.devices())
import jaxlib
import jax.numpy as jnp
import flax
import flax.linen as nn
import optax
from typing import Tuple, Callable, Any, Dict, Optional
import numpy.typing as npt
import copy
import pathlib
import matplotlib.pyplot as plt
import time
import json
import ast
import netket as nk
import os
import glob
import sys

sys.path.append("/home/ihuarte/Escritorio/Ivan/NNs")

# os.chdir("/home/ihuarte/Escritorio/Ivan/NN/")

from VA_project.model.model import J1J2Square
from VA_project.engine.runners import Runner

# from NN_utils import load_vstate
# from correlations import correlations_vstate

# from NNs.NN_module.ST_utils import compare_params, masked_optimizer
from frozendict import deepfreeze
from NN_module.ST_utils import print_tree

[CpuDevice(id=0)]


In [9]:
configurations = [
    "/home/ihuarte/Escritorio/Ivan/NNs/config_Hydra.json",
    "/home/ihuarte/Escritorio/Ivan/NNs/config_Hydra_NN.json",
]
with open(configurations[0], "r") as f:
    config_hydra = json.load(f)
with open(configurations[1], "r") as f:
    config_nn = json.load(f)

config = {**config_hydra, **config_nn}

# storage = config["storage"]
# symm_wrappers = config["symm_wrappers"]
# arch_evolution = config[config["selection"]]
key = jax.random.PRNGKey(0)
x = jnp.ones((1,16))

from NN_module.NN.Hydra import Hydra
hydra = Hydra(config, **{"lattice_size": [4,4]})



********************* ARCHITECTURE INFO *********************

├── ModulusNet (MLP)  --->  (0)
└── PhaseNet (CNN)  --->  (1)


NN stats: 10453 parameters (0.07975006103515625 MB)

*************************************************************


In [10]:
model_0 = hydra.model
params_0 = model_0.init(key, x)['params']


In [11]:
hydra.arch_evol(params_0)

********************* ARCHITECTURE INFO *********************

├── ModulusNet   --->  (0)
│   ├── Trans_0 (ViT2D)  --->  (00)
│   └── Trans_1 (MLP)  --->  (01)
└── PhaseNet   --->  (1)
    ├── Seq_0 (CNN)  --->  (10)
    └── ZZ (CvT)  --->  (11)


NN stats: 27437 parameters (0.17331695556640625 MB)

*************************************************************


In [12]:
model_1 = hydra.model
params_1 = model_1.init(key, x)['params']

In [ ]:

def get_subtree(tree, path):
    for key in path:
        tree = tree[key]
    return tree


def set_subtree(tree, path, new_subtree):
    if len(path) == 0:
        return new_subtree
    key = path[0]
    return {**tree, key: set_subtree(tree[key], path[1:], new_subtree)}

def flatten_with_paths(tree):
    out = []
    def visit(t, path):
        if isinstance(t, dict):
            for k, v in t.items():
                visit(v, path + (k,))
        elif isinstance(t, (list, tuple)):
            for i, v in enumerate(t):
                visit(v, path + (i,))
        else:
            out.append((path, t))
    visit(tree, ())
    return out

def find_leaf_matches(p0, p1):
    flat0 = flatten_with_paths(p0)
    flat1 = flatten_with_paths(p1)

    matches = []
    for path0, x in flat0:
        for path1, y in flat1:
            if x.shape == y.shape and x.dtype == y.dtype:
                matches.append((path0, path1))
    return matches

def greedy_transplant(old, new):
    matches = find_leaf_matches(old, new)
    used_old = set()
    used_new = set()
    mapping = []

    for p0, p1 in matches:
        if p0 not in used_old and p1 not in used_new:
            mapping.append((p0, p1))
            used_old.add(p0)
            used_new.add(p1)
    return mapping

In [14]:
new_params = params_1.copy()

In [6]:
import jax.numpy as jnp

def eon_change(i_eon, i_era, i_per, n_eons, n_eras, n_pers):

    eon_bool = i_eon != n_eons - 1
    era_bool = i_era == n_eras - 1
    per_bool = i_per == n_pers - 1

    return eon_bool & era_bool & per_bool

E, e, p = 2, 4, 3

eon =  jnp.arange(E)
era = jnp.arange(e)
period = jnp.arange(p)

N=E*e*p
for i in range(N):
    i_eon, i_era, i_per = jnp.unravel_index(i , (E, e, p))
    change = eon_change(i_eon, i_era, i_per, E, e, p)
    print(f"eon: {i_eon}/{E}   era:{i_era}/{e}  per: {i_per}/{p}   ¿¿Cambio?? ---> {change}")
    print("\n") if change else None


eon: 0/2   era:0/4  per: 0/3   ¿¿Cambio?? ---> False
eon: 0/2   era:0/4  per: 1/3   ¿¿Cambio?? ---> False
eon: 0/2   era:0/4  per: 2/3   ¿¿Cambio?? ---> False
eon: 0/2   era:1/4  per: 0/3   ¿¿Cambio?? ---> False
eon: 0/2   era:1/4  per: 1/3   ¿¿Cambio?? ---> False
eon: 0/2   era:1/4  per: 2/3   ¿¿Cambio?? ---> False
eon: 0/2   era:2/4  per: 0/3   ¿¿Cambio?? ---> False
eon: 0/2   era:2/4  per: 1/3   ¿¿Cambio?? ---> False
eon: 0/2   era:2/4  per: 2/3   ¿¿Cambio?? ---> False
eon: 0/2   era:3/4  per: 0/3   ¿¿Cambio?? ---> False
eon: 0/2   era:3/4  per: 1/3   ¿¿Cambio?? ---> False
eon: 0/2   era:3/4  per: 2/3   ¿¿Cambio?? ---> True


eon: 1/2   era:0/4  per: 0/3   ¿¿Cambio?? ---> False
eon: 1/2   era:0/4  per: 1/3   ¿¿Cambio?? ---> False
eon: 1/2   era:0/4  per: 2/3   ¿¿Cambio?? ---> False
eon: 1/2   era:1/4  per: 0/3   ¿¿Cambio?? ---> False
eon: 1/2   era:1/4  per: 1/3   ¿¿Cambio?? ---> False
eon: 1/2   era:1/4  per: 2/3   ¿¿Cambio?? ---> False
eon: 1/2   era:2/4  per: 0/3   ¿¿Cambio?? ---

In [19]:
new_c2p = hydra.get_code2path(new_params)


for (params_name, old_code, new_code) in hydra.load:

    old_params = hydra.params_history[params_name]
    old_c2p = hydra.get_code2path(old_params)

    old_path_block = old_c2p[old_code]
    new_path_block = new_c2p[new_code]

    old_params_block = get_subtree(old_params, old_path_block)
    new_params_block = get_subtree(new_params, new_path_block)

    trasplant = greedy_transplant(old_params_block, new_params_block)
    print(trasplant)
    print("\n\n")

    for (old_rel, new_rel) in trasplant:
        old_abs = old_path_block + old_rel
        new_abs = new_path_block + new_rel

        val = get_subtree(old_params, old_abs)
        new_params = set_subtree(new_params, new_abs, val)


[(('MLPWorker_0', 'Dense_0', 'kernel'), ('MLPWorker_0', 'Dense_0', 'kernel')), (('MLPWorker_0', 'Dense_0', 'bias'), ('MLPWorker_0', 'Dense_0', 'bias')), (('MLPWorker_0', 'LayerNorm_0', 'scale'), ('MLPWorker_0', 'LayerNorm_0', 'scale')), (('MLPWorker_0', 'LayerNorm_0', 'bias'), ('MLPWorker_0', 'LayerNorm_0', 'bias')), (('MLPWorker_0', 'Dense_1', 'kernel'), ('MLPWorker_0', 'Dense_1', 'kernel')), (('MLPWorker_0', 'Dense_1', 'bias'), ('MLPWorker_0', 'Dense_1', 'bias')), (('MLPWorker_0', 'LayerNorm_1', 'scale'), ('MLPWorker_0', 'LayerNorm_1', 'scale')), (('MLPWorker_0', 'LayerNorm_1', 'bias'), ('MLPWorker_0', 'LayerNorm_1', 'bias')), (('MLPWorker_0', 'Dense_2', 'kernel'), ('MLPWorker_0', 'Dense_2', 'kernel')), (('MLPWorker_0', 'Dense_2', 'bias'), ('MLPWorker_0', 'Dense_2', 'bias')), (('MLPWorker_0', 'LayerNorm_2', 'scale'), ('MLPWorker_0', 'LayerNorm_2', 'scale')), (('MLPWorker_0', 'LayerNorm_2', 'bias'), ('MLPWorker_0', 'LayerNorm_2', 'bias')), (('MLPWorker_0', 'Dense_3', 'kernel'), ('MLPW

In [11]:
from NN_module.ST_utils import compare_params
compare_params(params_1, new_params)

Ha cambiado: (True) //  No ha cambiado: (False) 


ModulusNet
   Trans_0
      ViT2DTokenize_0
         ViT2DWorker_0
            CoreBlock_0
               LayerNorm_0
                  bias: False
                  scale: False
               MultiHeadPositionalAttention_0
                  PositionalHead_0
                     AffinityPosWeight_0
                        alpha_delta_nosymm: False
                     Dense_0
                        kernel: False
                  PositionalHead_1
                     AffinityPosWeight_0
                        alpha_delta_nosymm: False
                     Dense_0
                        kernel: False
               MultiLayerPerceptron_0
                  Dense_0
                     bias: False
                     kernel: False
                  LayerNorm_0
                     bias: False
                     scale: False
            CoreBlock_1
               LayerNorm_0
                  bias: False
                  scale: Fal

In [16]:
new_c2p = hydra.get_code2path(new_params)

In [17]:
new_c2p['10']

('PhaseNet', 'Seq_0')

In [ ]:
from NN_module.schedule.schedule import Schedule
import optax

setup = {
    "print_arch": True,
    "learning_rate": {
        "epochs_struct": [[[50], [100], [200]]],
        "modes_struct": [[[["A"]], [["1", "110"]], [["10000"]]]],
        "lr_struct": [
            [[[0.01, 0.05]], [["lin(0.001, 0.1)", 0.3]], [["exp(0.1, 0.001)"]]]
        ],
        "repeat": [],
        "rescale": 1.0,
    },
    "architecture": {},
    "sampler": {},
}


sch = Schedule(setup, vstate.parameters)
sch.total_epochs
opt = optax.sgd
for period, info in sch.schedule():
    print(f"{info}")
    print(f"{period} {type(period)}\n")
    sch.transform_optimizer(vstate.parameters, opt, info, period)
    print("\n")

In [ ]:
import flax.linen as nn

arch_name = type(model).__name__
subarch_names = [
    name
    for name in model.__dict__.keys()
    if isinstance(model.__dict__[name], nn.Module)
]
arch_name, subarch_names

In [ ]:
model.__dict__["Trans"][0].__class__.__name__

In [ ]:
def is_subsequence(a, b):
    n, m = len(a), len(b)
    for i in range(m - n + 1):
        if b[i : i + n] == a:
            return True
    return False


a = (1, 2, 3)
b = (1, 2, 4, 3, 4, 5, 6)
is_subsequence(a, b)

In [ ]:
factory.setup